In [ ]:
# ==============================================================================
# MINI PROJECT 1: INDIAN STOCK MARKET ANALYSIS & RETURN PREDICTION (2000-2020)
# ==============================================================================
# OBJECTIVE:
#   1. Ingest historical Indian equities/indices data (Open, High, Low, Close, Volume).
#   2. Compute financial return metrics (Daily Returns, Log Returns, Direction).
#   3. Feature Engineering: Rolling volatility, SMAs (10, 50, 200), RSI, Price Momentum.
#   4. Analyze Market Ups & Downs (Bullish vs. Bearish regimes).
#   5. Train Machine Learning Models (Random Forest Regressor & Gradient Boosting)
#      using strictly forward-chaining temporal splits (zero lookahead leakage).
#   6. Evaluate Return Predictions using RMSE, MAE, R², and Directional Accuracy.
#
# TIME & SPACE COMPLEXITIES:
#   - Feature Engineering (Rolling Windows): O(N * W) where W = max window length.
#   - Random Forest Regressor:
#       * Training: O(B * N_train * log(N_train) * D) where B = estimators, D = features.
#       * Inference: O(B * depth) per query.
#       * Space: O(B * Total_Tree_Nodes).
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------------------------------------------
# Step 1: Load and Pre-process Historical Stock Data
# ------------------------------------------------------------------------------
# Note: Ensure your CSV contains standard OHLCV columns: Date, Open, High, Low, Close, Volume
df = pd.read_csv('stock_data.csv')

# Standardize column naming conventions
df.columns = df.columns.str.strip().str.capitalize()

# Parse datetime and enforce chronological ordering
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

# ------------------------------------------------------------------------------
# Step 2: Financial Engineering (Returns, Volatility & Regime Detection)
# ------------------------------------------------------------------------------
# 1. Simple Daily Return: R_t = (P_t - P_{t-1}) / P_{t-1}
df['Return'] = df['Close'].pct_change()

# 2. Logarithmic Return: r_t = ln(P_t / P_{t-1})
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# 3. Market Regime Indicator (Up = 1, Down = 0)
df['Market_Regime'] = (df['Return'] > 0).astype(int)

# 4. Rolling Simple Moving Averages (Trend indicators)
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# 5. Annualized Rolling Volatility (252 trading sessions)
df['Volatility_20d'] = df['Log_Return'].rolling(window=20).std() * np.sqrt(252)

# 6. Relative Strength Index (RSI - 14 sessions)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-9)
df['RSI_14'] = 100 - (100 / (1 + rs))

# 7. Target Variable: Next Session's Percentage Return (Shifted backward by 1)
df['Target_Next_Return'] = df['Return'].shift(-1)

# Drop initial window NaN values and the final unlabelled target record
df.dropna(inplace=True)

# ------------------------------------------------------------------------------
# Step 3: Analysis of Ups and Downs (Market Distribution)
# ------------------------------------------------------------------------------
total_sessions = len(df)
up_sessions = (df['Market_Regime'] == 1).sum()
down_sessions = (df['Market_Regime'] == 0).sum()

print("=" * 60)
print("INDIAN STOCK MARKET HISTORICAL REGIME ANALYSIS (2000-2020)")
print("=" * 60)
print(f"Total Analyzed Trading Days : {total_sessions}")
print(f"Upward Sessions (Gainers)   : {up_sessions} ({up_sessions / total_sessions * 100:.2f}%)")
print(f"Downward Sessions (Losers)  : {down_sessions} ({down_sessions / total_sessions * 100:.2f}%)")
print(f"Mean Annualized Volatility  : {df['Volatility_20d'].mean() * 100:.2f}%")
print(f"Maximum 1-Day Price Surge   : {df['Return'].max() * 100:.2f}%")
print(f"Maximum 1-Day Price Plunge  : {df['Return'].min() * 100:.2f}%")
print("=" * 60)

# ------------------------------------------------------------------------------
# Step 4: Time-Series Train/Test Split (Strictly Chronological)
# ------------------------------------------------------------------------------
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 
    'Return', 'Log_Return', 'SMA_10', 'SMA_50', 
    'SMA_200', 'Volatility_20d', 'RSI_14'
]

X = df[feature_cols]
y = df['Target_Next_Return']

# 80% Chronological Train, 20% Chronological Test
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# ------------------------------------------------------------------------------
# Step 5: Model Training (Random Forest & Gradient Boosting)
# ------------------------------------------------------------------------------
# Model A: Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

# Model B: Gradient Boosting Regressor
gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
gb_reg.fit(X_train, y_train)
y_pred_gb = gb_reg.predict(X_test)

# ------------------------------------------------------------------------------
# Step 6: Evaluation Metrics & Directional Prediction
# ------------------------------------------------------------------------------
def evaluate_forecast(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # Directional Accuracy: True if predicted return sign matches actual sign
    dir_acc = np.mean((y_true > 0) == (y_pred > 0)) * 100
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2_Score': r2,
        'Directional_Accuracy (%)': dir_acc
    }

eval_results = pd.DataFrame([
    evaluate_forecast(y_test, y_pred_rf, "Random Forest Regressor"),
    evaluate_forecast(y_test, y_pred_gb, "Gradient Boosting Regressor")
])

print("\n--- Model Performance Comparison ---")
print(eval_results.to_string(index=False))

# ------------------------------------------------------------------------------
# Step 7: Visualizing Predictions vs Actual Returns
# ------------------------------------------------------------------------------
plt.figure(figsize=(14, 5))
plt.plot(df['Date'].iloc[split_idx:split_idx+150], y_test.iloc[:150], label='Actual Returns', color='black', alpha=0.6)
plt.plot(df['Date'].iloc[split_idx:split_idx+150], y_pred_rf[:150], label='Predicted Returns (RF)', color='crimson', linestyle='--')
plt.title('Stock Return Prediction (Test Sample Horizon)')
plt.xlabel('Date')
plt.ylabel('Daily Return Percentage')
plt.legend()
plt.grid(True)
plt.show()